In [1]:
import pandas as pd
import re
import numpy as np

# Goal: Build a pipeline for data prep

In [2]:
## load in data
dat = pd.read_csv('full_fight_features.csv')
print(dat.shape)
dat.head()

(8729, 1036)


,Unnamed: 0,fight_id,event_id,date,location,results.fight_winner,results.fight_loser,results.method,results.round,results.time,...,sig_str.ground_attempted_received_5fight_MAX_fighter_b,sig_str.ground_landed_received_2fight_AVG_fighter_b,sig_str.ground_landed_received_2fight_MIN_fighter_b,sig_str.ground_landed_received_2fight_MAX_fighter_b,sig_str.ground_landed_received_3fight_AVG_fighter_b,sig_str.ground_landed_received_3fight_MIN_fighter_b,sig_str.ground_landed_received_3fight_MAX_fighter_b,sig_str.ground_landed_received_5fight_AVG_fighter_b,sig_str.ground_landed_received_5fight_MIN_fighter_b,sig_str.ground_landed_received_5fight_MAX_fighter_b
0,0,11cc331fa40ede63,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Marlon Vera,Dominick Cruz,KO/TKO,4,2:17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,abd50fad1e7b030b,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Priscila Cachoeira,Ariane da Silva,KO/TKO,1,1:05,...,6.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
2,2,f858f91453555291,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Gabriel Benitez,Charlie Ontiveros,KO/TKO,1,3:33,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,5f31f3e32669f5b9,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Angela Hill,Loopy Godinez,Decision - Unanimous,3,5:00,...,7.0,0.0,0.0,0.0,2.0,0.0,6.0,1.5,0.0,6.0
4,4,0823f97be9c3a62e,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Tyson Nam,Ode Osbourne,KO/TKO,1,2:59,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
# ## Create target variable
# def feature_preprocess(dat, non_modeling_cols):

#     ## create binary target value, did fighter_a win? 
#     dat = dat['fighter_a_won'] = dat['results.fight_winner'] == dat['totals.name_fighter_a']

#     ## impute nulls with 0
#     dat = dat.fillna(0, inplace=True)

#     ## define cols
#     y_col = ['fighter_a_won']
#     x_cols = [col for col in dat.columns if col not in (non_modeling_cols + y_col)]

#     ## convert time col from minutes:seconds to secods -- WILL DO upsteam
#     x_cols = [col for col in x_cols if 'control' not in col]

#     ## format float (pct) columns
#     float_cols = [col for col in dat.columns if 'pct' in col]
#     dat[float_cols] = dat[float_cols].replace({'%':''}, regex=True).replace({'---':0}, regex=True)

#     ## format integer features
#     int_cols = [col for col in x_cols if col not in float_cols]

#     ## Combine lists into a single dictionary mapping using comprehension
#     dtype_mapping = {col: int for col in int_cols} | {
#         col: float for col in float_cols
#         }

#     dat_processed = dat.astype(dtype_mapping)

#     return dat_processed



In [4]:
from preprocess_functions import feature_preprocess

In [6]:
non_modeling_cols = ['Unnamed: 0', 'fight_id', 'event_id', 'date', 'location', 'results.fight_winner', 'results.fight_loser',
                     'results.method', 'results.round','results.time', 'results.time_format','results.referee',
                     'results.details','results.fight_name','id_a','id_b','fighter_fight_id.2_fighter_a', 'totals.id_fighter_a',
                     'totals.id_received_fighter_a', 'totals.name_fighter_a','totals.name_received_fighter_a','error_fighter_a',
                     'fighter_fight_id.1_fighter_a','fighter_fight_id.2_fighter_b', 'totals.id_fighter_b','totals.id_received_fighter_b',
                     'totals.name_fighter_b', 'totals.name_received_fighter_b', 'error_fighter_b', 'fighter_fight_id.1_fighter_b']

dat_processed = feature_preprocess(dat, non_modeling_cols)
dat_processed.head()

,Unnamed: 0,fight_id,event_id,date,location,results.fight_winner,results.fight_loser,results.method,results.round,results.time,...,sig_str.ground_landed_received_2fight_AVG_fighter_b,sig_str.ground_landed_received_2fight_MIN_fighter_b,sig_str.ground_landed_received_2fight_MAX_fighter_b,sig_str.ground_landed_received_3fight_AVG_fighter_b,sig_str.ground_landed_received_3fight_MIN_fighter_b,sig_str.ground_landed_received_3fight_MAX_fighter_b,sig_str.ground_landed_received_5fight_AVG_fighter_b,sig_str.ground_landed_received_5fight_MIN_fighter_b,sig_str.ground_landed_received_5fight_MAX_fighter_b,fighter_a_won
0,0,11cc331fa40ede63,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Marlon Vera,Dominick Cruz,KO/TKO,4,2:17,...,0,0,0,0,0,0,0,0,0,True
1,1,abd50fad1e7b030b,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Priscila Cachoeira,Ariane da Silva,KO/TKO,1,1:05,...,4,4,4,4,4,4,4,4,4,True
2,2,f858f91453555291,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Gabriel Benitez,Charlie Ontiveros,KO/TKO,1,3:33,...,0,0,0,0,0,0,0,0,0,True
3,3,5f31f3e32669f5b9,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Angela Hill,Loopy Godinez,Decision - Unanimous,3,5:00,...,0,0,0,2,0,6,1,0,6,True
4,4,0823f97be9c3a62e,a23e63184c65f5b8,"August 13, 2022","San Diego, California, USA",Tyson Nam,Ode Osbourne,KO/TKO,1,2:59,...,0,0,0,0,0,0,0,0,0,False


In [7]:
dat_processed[['results.fight_winner','totals.name_fighter_a','fighter_a_won']].head()


,results.fight_winner,totals.name_fighter_a,fighter_a_won
0,Marlon Vera,Marlon Vera,True
1,Priscila Cachoeira,Priscila Cachoeira,True
2,Gabriel Benitez,Gabriel Benitez,True
3,Angela Hill,Angela Hill,True
4,Tyson Nam,Ode Osbourne,False
